In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# Import Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# MODULE: DATASET PROFILER
# ==========================================
def profile_dataset(file_path):
    """
    Loads and outputs a comprehensive analysis of the wdbc dataset structure.
    """
    # Define columns based on WDBC documentation
    column_names = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
    df = pd.read_csv(file_path, header=None, names=column_names)
    
    print("=" * 50)
    print("        MODULE: DATASET DETAILS & PROFILE        ")
    print("=" * 50)
    
    # 1. Structural Details
    print(f"• Total Rows (Instances): {df.shape[0]}")
    print(f"• Total Columns (Features): {df.shape[1]}")
    
    # 2. Class Balance Breakdown
    class_counts = df['diagnosis'].value_counts()
    class_pct = df['diagnosis'].value_counts(normalize=True) * 100
    print("\n• Target Variable Breakdown ('diagnosis'):")
    for label, count in class_counts.items():
        name = "Malignant (Cancerous)" if label == 'M' else "Benign (Non-Cancerous)"
        print(f"  - {label} ({name}): {count} samples ({class_pct[label]:.2f}%)")
        
    # 3. Missing Value / Integrity Audit
    missing_sum = df.isnull().sum().sum()
    print(f"\n• Missing Values Check: {missing_sum} missing values found.")
    
    # 4. Feature Space Layout
    print("\n• Attribute Structural Layout:")
    print("  - Column 1: Patient ID (To be dropped during training)")
    print("  - Column 2: Diagnosis Target Label (M / B)")
    print("  - Columns 3-12:  Mean Features (Mean of cell nuclei dimensions)")
    print("  - Columns 13-22: Standard Error (SE) Features")
    print("  - Columns 23-32: Worst / Largest Features (Extreme cell dimensions)")
    
    # 5. Statistical Sample View
    print("\n• Sample Statistical View of Core Mean Features:")
    mean_cols = [f'feature_{i}' for i in range(1, 6)] # Display first 5 features for brevity
    display(df[mean_cols].describe().round(3))
    print("=" * 50 + "\n")
    
    return df

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# 1. Execute Dataset Profiler Module
df_raw = profile_dataset('wdbc.data')

# 2. Preprocessing Data
df_ml = df_raw.drop(columns=['id'])
le = LabelEncoder()
df_ml['diagnosis'] = le.fit_transform(df_ml['diagnosis'])

X = df_ml.drop(columns=['diagnosis'])
y = df_ml['diagnosis']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Model Training & Evaluation Setup
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbor": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes (Gaussian)": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=100, random_state=42)
}

results_dict = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    results_dict[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC Score": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC Score": matthews_corrcoef(y_test, y_pred)
    }

# 4. Display Final Model Evaluation
performance_df = pd.DataFrame(results_dict).T
print("==== ML Models Evaluation Summary ====")
display(performance_df.round(4))


        MODULE: DATASET DETAILS & PROFILE        
• Total Rows (Instances): 569
• Total Columns (Features): 32

• Target Variable Breakdown ('diagnosis'):
  - B (Benign (Non-Cancerous)): 357 samples (62.74%)
  - M (Malignant (Cancerous)): 212 samples (37.26%)

• Missing Values Check: 0 missing values found.

• Attribute Structural Layout:
  - Column 1: Patient ID (To be dropped during training)
  - Column 2: Diagnosis Target Label (M / B)
  - Columns 3-12:  Mean Features (Mean of cell nuclei dimensions)
  - Columns 13-22: Standard Error (SE) Features
  - Columns 23-32: Worst / Largest Features (Extreme cell dimensions)

• Sample Statistical View of Core Mean Features:


,feature_1,feature_2,feature_3,feature_4,feature_5
count,569.000,569.000,569.000,569.000,569.000
mean,14.127,19.290,91.969,654.889,0.096
std,3.524,4.301,24.299,351.914,0.014
min,6.981,9.710,43.790,143.500,0.053
25%,11.700,16.170,75.170,420.300,0.086
50%,13.370,18.840,86.240,551.100,0.096
75%,15.780,21.800,104.100,782.700,0.105
max,28.110,39.280,188.500,2501.000,0.163



==== ML Models Evaluation Summary ====


,Accuracy,AUC Score,Precision,Recall,F1 Score,MCC Score
Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
K-Nearest Neighbor,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
Naive Bayes (Gaussian),0.9211,0.9891,0.9231,0.8571,0.8889,0.8292
Random Forest (Ensemble),0.9737,0.9929,1.0000,0.9286,0.9630,0.9442
